# SPI walltime benchmark — modular & family analysis

Per-SPI amortized walltime from the in-repo bench suite
(`bench/results/cells/*.json` merged with `bench/results/analysis/long_costs.csv`).
Run `python -m bench.analyse_cells` once first: `long_costs.csv` is a derived artefact and is not committed.

Each SPI is tagged two ways:

- **family** (`basic`, `causal`, `distance`, `infotheory`, `misc`, `spectral`, `wavelet`) — the original pyspi groupings.
- **module** (`M01`..`M14` from Cliff et al. 2023, plus `Mxx` for SPIs added after the paper) — Cliff's modular/literature-based grouping.

The 1 unlabelled SPI (`ids`) is bucketed into `Mxx` as a post-paper addition.

Focus cell for single-cell figures: **M=16, T=800**. Kept-set Jaccard@p90 across all 22 cells is 0.976, so SPI ordering is highly stable — the focus cell mostly affects the dynamic range visible on the log axis, not the conclusions. Change `FOCUS_M, FOCUS_T` in the loader to swap.


In [ ]:
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 110

NB_DIR = Path.cwd()
REPO_ROOT = next(p for p in [NB_DIR, *NB_DIR.parents] if (p / "pyspi").exists())
CELLS_DIR = REPO_ROOT / "bench" / "results" / "cells"
ANALYSIS_DIR = REPO_ROOT / "bench" / "results" / "analysis"

FOCUS_M, FOCUS_T = 16, 800

MODULE_ORDER = [f"M{n:02d}" for n in range(1, 15)] + ["Mxx"]
FAMILY_ORDER = ["basic", "causal", "distance", "infotheory", "misc", "spectral", "wavelet"]

# Turbo gradient over modules: M01..M14 sweep the rainbow, Mxx lands at the warm end.
_grad = px.colors.sample_colorscale("Turbo", np.linspace(0.05, 0.95, len(MODULE_ORDER)))
MODULE_COLORS = dict(zip(MODULE_ORDER, _grad))

# Family palette: match px.violin's default discrete cycle so 3b and 3c agree.
FAMILY_COLORS = {f: px.colors.qualitative.Plotly[i] for i, f in enumerate(FAMILY_ORDER)}


## 1. Load benchmarking data

In [ ]:
def _module(labels):
    """First M-tag in a label list; SPIs without one are treated as post-paper (Mxx)."""
    for l in labels:
        if len(l) == 3 and l[0] == "M" and (l[1:].isdigit() or l[1:] == "XX"):
            return "Mxx" if l == "MXX" else l
    return "Mxx"

rows = []
for f in sorted(CELLS_DIR.glob("*.json")):
    d = json.loads(f.read_text())
    if "spi_seconds" not in d or "error" in d:
        continue
    for spi, meta in d["spi_seconds"].items():
        rows.append({
            "M": d["M"], "T": d["T"],
            "spi": spi,
            "family": meta.get("category"),
            "module": _module(meta.get("labels", [])),
            "raw_s": float(meta["mean"]),
        })
raw_df = pd.DataFrame(rows)

amo_df = pd.read_csv(ANALYSIS_DIR / "long_costs.csv")
df = raw_df.merge(
    amo_df[["M", "T", "identifier", "amortized_s", "cache_namespace"]],
    left_on=["M", "T", "spi"], right_on=["M", "T", "identifier"], how="left",
).drop(columns="identifier")
df["cell"] = "M=" + df["M"].astype(str) + ", T=" + df["T"].astype(str)

n_cells = df[["M", "T"]].drop_duplicates().shape[0]
print(f"{len(df):,} rows | {df.spi.nunique()} SPIs | {n_cells} cells | "
      f"{df.module.nunique()} modules | {df.family.nunique()} families")
df.head()


In [ ]:
# Cell-level summary (matches bench/results/analysis/cell_summary.csv)
cell_summary = (df.groupby(["M", "T"])
                  .agg(n_spis=("spi", "nunique"),
                       sum_amortized_s=("amortized_s", "sum"),
                       median_amortized_s=("amortized_s", "median"),
                       max_amortized_s=("amortized_s", "max"))
                  .round(3)
                  .reset_index())
cell_summary


## 2. Figure group 0 — Per-SPI walltime distribution

Histogram + KDE on log-walltime, first for the focus cell, then pooled across all cells. The M×T heatmap below shows the planning number — total amortized walltime per cell, with extrapolation marked `*` where the cell was not measured.

### 2a. Focus-cell histogram (M=16, T=800)

In [ ]:
focus = df[(df["M"] == FOCUS_M) & (df["T"] == FOCUS_T) & (df["amortized_s"] > 0)]

fig, ax = plt.subplots(figsize=(11, 4.5))
sns.histplot(data=focus, x="amortized_s", stat="density", bins=50,
             log_scale=(True, False), kde=True, color="steelblue",
             line_kws={"lw": 2}, ax=ax)
ax.set_xlabel("per-SPI amortized walltime (s, log scale)")
ax.set_ylabel("density")
ax.set_title(f"Focus cell M={FOCUS_M}, T={FOCUS_T} — {len(focus)} SPIs")
plt.tight_layout()
plt.show()


### 2b. Pooled across all 22 cells

In [ ]:
pos = df[df["amortized_s"] > 0]
n_cells = pos[["M", "T"]].drop_duplicates().shape[0]

fig, ax = plt.subplots(figsize=(11, 4.5))
sns.histplot(data=pos, x="amortized_s", stat="density", bins=60,
             log_scale=(True, False), kde=True, color="indianred",
             line_kws={"lw": 2}, ax=ax)
ax.set_xlabel("per-SPI amortized walltime (s, log scale)")
ax.set_ylabel("density")
ax.set_title(f"Pooled — {pos.spi.nunique()} SPIs × {n_cells} cells")
plt.tight_layout()
plt.show()


### 2c. M × T heatmap — total amortized cell walltime

Cells with no measured run are filled from a log-log fit `log(total) = a + p·log(M) + q·log(T)` over the observed cells and marked `*`.

In [ ]:
M_VALS = sorted(df["M"].unique())
T_VALS = sorted(df["T"].unique())

observed = (df.groupby(["M", "T"])["amortized_s"].sum()
              .reindex(pd.MultiIndex.from_product([M_VALS, T_VALS], names=["M", "T"])))

# log-log fit on observed cell totals
obs = observed.dropna().reset_index()
X = np.column_stack([np.ones(len(obs)), np.log(obs["M"]), np.log(obs["T"])])
y = np.log(obs["amortized_s"])
beta, *_ = np.linalg.lstsq(X, y, rcond=None)
predict = lambda M, T: float(np.exp(beta[0] + beta[1]*np.log(M) + beta[2]*np.log(T)))

mat = np.zeros((len(M_VALS), len(T_VALS)))
is_extrap = np.zeros_like(mat, dtype=bool)
for i, M in enumerate(M_VALS):
    for j, T in enumerate(T_VALS):
        v = observed.loc[(M, T)]
        if pd.isna(v):
            mat[i, j] = predict(M, T); is_extrap[i, j] = True
        else:
            mat[i, j] = v

fig, ax = plt.subplots(figsize=(9, 4.8))
im = ax.imshow(np.log10(mat), aspect="auto", cmap="viridis", origin="lower")
ax.set_xticks(range(len(T_VALS))); ax.set_xticklabels(T_VALS)
ax.set_yticks(range(len(M_VALS))); ax.set_yticklabels(M_VALS)
ax.set_xlabel("T"); ax.set_ylabel("M")
ax.set_title("Total amortized cell walltime (s) — * = extrapolated from log-log fit\n"
             f"fit: log(total) = {beta[0]:.2f} + {beta[1]:.2f}·log(M) + {beta[2]:.2f}·log(T)")

vmid = (np.log10(mat).min() + np.log10(mat).max()) / 2
for i in range(len(M_VALS)):
    for j in range(len(T_VALS)):
        s = f"{mat[i, j]:.0f}" if mat[i, j] >= 100 else f"{mat[i, j]:.1f}"
        if is_extrap[i, j]:
            s += "*"
        ax.text(j, i, s, ha="center", va="center", fontsize=9,
                color="white" if np.log10(mat[i, j]) < vmid else "black")
fig.colorbar(im, ax=ax, label="log10(walltime, s)")
plt.tight_layout()
plt.show()


## 3. Figure group 1 — Per-module and per-family rainclouds

Half-violin + jittered dots, drawn at the focus cell so each SPI contributes exactly one point per panel. Hover for SPI identifier, family/module, and walltime. Y axis is log walltime; X axis is the grouping.

### 3a. Per-module raincloud — animated across all cells

Slider/play scrubs through every (M, T) cell. Each frame is one cell; one half-violin + jittered SPI dots per module.

In [ ]:
df_pos = df.loc[df["amortized_s"] > 0].copy()
df_cell = df_pos.assign(
    log_walltime_s=np.log10(df_pos["amortized_s"]),
)
cell_order = sorted(df_cell["cell"].unique(),
                    key=lambda s: (int(s.split(",")[0][2:]), int(s.split("T=")[1])))
module_order = [m for m in MODULE_ORDER if m in df_cell["module"].values]

# Pad (cell, module) combos so every animation frame keeps all modules on axis.
fill = []
for cell in cell_order:
    present = set(df_cell.loc[df_cell["cell"] == cell, "module"])
    for mod in module_order:
        if mod not in present:
            fill.append({"cell": cell, "module": mod, "spi": "",
                         "amortized_s": float("nan"), "log_walltime_s": float("nan"),
                         "M": None, "T": None})
if fill:
    df_cell = pd.concat([df_cell, pd.DataFrame(fill)], ignore_index=True)

exp_min = int(np.floor(df_cell["log_walltime_s"].min()))
exp_max = int(np.ceil(df_cell["log_walltime_s"].max()))
tickvals = list(range(exp_min, exp_max + 1))
ticktext = [f"{10**e:g}" for e in tickvals]

fig = px.violin(
    df_cell,
    y="log_walltime_s", x="module", color="module",
    color_discrete_map=MODULE_COLORS,
    animation_frame="cell",
    category_orders={"cell": cell_order, "module": module_order},
    hover_data={
        "spi": True, "M": True, "T": True,
        "amortized_s": ":.4f", "family": True,
        "module": False, "cell": False, "log_walltime_s": False,
    },
    points="all",
    height=500,
    width=900,
    range_y=[df_cell["log_walltime_s"].min() - 0.2,
             df_cell["log_walltime_s"].max() + 0.2],
)

style = {
    "side": "positive",
    "pointpos": -0.35,
    "jitter": 0.22,
    "width": 0.55,
    "meanline_visible": False,
    "box_visible": False,
    "marker": {"size": 6, "opacity": 0.72},
    "opacity": 0.72,
}
for trace in fig.data:
    trace.update(**style)
for frame in fig.frames:
    for trace in frame.data:
        trace.update(**style)

fig.update_layout(
    yaxis={
        "title": "per-SPI amortized walltime (s, log scale)",
        "tickmode": "array",
        "tickvals": tickvals,
        "ticktext": ticktext,
    },
    xaxis_title="module",
    title="Per-module raincloud per (M, T) cell — use slider / play button",
    legend_title_text="module",
    margin={"l": 80, "r": 20, "t": 60, "b": 80},
)
fig.show()


### 3a-snapshot. Per-module raincloud — focus cell (M=16, T=800)

In [ ]:
focus_pos = df.loc[(df["M"] == FOCUS_M) & (df["T"] == FOCUS_T) & (df["amortized_s"] > 0)].copy()
focus_pos["log_walltime_s"] = np.log10(focus_pos["amortized_s"])
modules_present = [m for m in MODULE_ORDER if m in focus_pos["module"].values]

exp_min = int(np.floor(focus_pos["log_walltime_s"].min()))
exp_max = int(np.ceil(focus_pos["log_walltime_s"].max()))
tickvals = list(range(exp_min, exp_max + 1))
ticktext = [f"{10**e:g}" for e in tickvals]

fig = px.violin(
    focus_pos,
    y="log_walltime_s", x="module", color="module",
    color_discrete_map=MODULE_COLORS,
    category_orders={"module": modules_present},
    hover_data={
        "spi": True, "M": True, "T": True,
        "amortized_s": ":.4f", "family": True,
        "module": False, "log_walltime_s": False,
    },
    points="all",
    height=500,
    width=900,
    range_y=[focus_pos["log_walltime_s"].min() - 0.2,
             focus_pos["log_walltime_s"].max() + 0.2],
)

style = {
    "side": "positive",
    "pointpos": -0.35,
    "jitter": 0.22,
    "width": 0.55,
    "meanline_visible": False,
    "box_visible": False,
    "marker": {"size": 6, "opacity": 0.72},
    "opacity": 0.72,
}
for trace in fig.data:
    trace.update(**style)

fig.update_layout(
    yaxis={
        "title": "per-SPI amortized walltime (s, log scale)",
        "tickmode": "array",
        "tickvals": tickvals,
        "ticktext": ticktext,
    },
    xaxis_title="module",
    title=f"Per-module raincloud — M={FOCUS_M}, T={FOCUS_T} snapshot",
    legend_title_text="module",
    margin={"l": 80, "r": 20, "t": 60, "b": 80},
)
fig.show()


### 3b. Per-family raincloud — animated across all cells

Slider/play scrubs through every (M, T) cell. Each frame is one cell; one half-violin + jittered SPI dots per family.

In [ ]:
df_pos = df.loc[df["amortized_s"] > 0].copy()
df_cell = df_pos.assign(
    log_walltime_s=np.log10(df_pos["amortized_s"]),
)
cell_order = sorted(df_cell["cell"].unique(),
                    key=lambda s: (int(s.split(",")[0][2:]), int(s.split("T=")[1])))
family_order = [f for f in FAMILY_ORDER if f in df_cell["family"].values]

# Pad (cell, family) combos so every animation frame keeps all families on axis.
fill = []
for cell in cell_order:
    present = set(df_cell.loc[df_cell["cell"] == cell, "family"])
    for fam in family_order:
        if fam not in present:
            fill.append({"cell": cell, "family": fam, "spi": "",
                         "amortized_s": float("nan"), "log_walltime_s": float("nan"),
                         "M": None, "T": None})
if fill:
    df_cell = pd.concat([df_cell, pd.DataFrame(fill)], ignore_index=True)

exp_min = int(np.floor(df_cell["log_walltime_s"].min()))
exp_max = int(np.ceil(df_cell["log_walltime_s"].max()))
tickvals = list(range(exp_min, exp_max + 1))
ticktext = [f"{10**e:g}" for e in tickvals]

fig = px.violin(
    df_cell,
    y="log_walltime_s", x="family", color="family",
    animation_frame="cell",
    category_orders={"cell": cell_order, "family": family_order},
    hover_data={
        "spi": True, "M": True, "T": True,
        "amortized_s": ":.4f", "family": False, "cell": False,
        "log_walltime_s": False,
    },
    points="all",
    height=500,
    width=900,
    range_y=[df_cell["log_walltime_s"].min() - 0.2,
             df_cell["log_walltime_s"].max() + 0.2],
)

style = {
    "side": "positive",
    "pointpos": -0.35,
    "jitter": 0.22,
    "width": 0.55,
    "meanline_visible": False,
    "box_visible": False,
    "marker": {"size": 6, "opacity": 0.72},
    "opacity": 0.72,
}
for trace in fig.data:
    trace.update(**style)
for frame in fig.frames:
    for trace in frame.data:
        trace.update(**style)

fig.update_layout(
    yaxis={
        "title": "per-SPI amortized walltime (s, log scale)",
        "tickmode": "array",
        "tickvals": tickvals,
        "ticktext": ticktext,
    },
    xaxis_title="family",
    title="Per-family raincloud per (M, T) cell — use slider / play button",
    legend_title_text="family",
    margin={"l": 80, "r": 20, "t": 60, "b": 80},
)
fig.show()


### 3b-snapshot. Per-family raincloud — focus cell (M=16, T=800)

In [ ]:
df_focus = df.loc[(df["M"] == FOCUS_M) & (df["T"] == FOCUS_T) & (df["amortized_s"] > 0)].copy()
df_focus["log_walltime_s"] = np.log10(df_focus["amortized_s"])
family_order = [f for f in FAMILY_ORDER if f in df_focus["family"].values]

exp_min = int(np.floor(df_focus["log_walltime_s"].min()))
exp_max = int(np.ceil(df_focus["log_walltime_s"].max()))
tickvals = list(range(exp_min, exp_max + 1))
ticktext = [f"{10**e:g}" for e in tickvals]

fig = px.violin(
    df_focus,
    y="log_walltime_s", x="family", color="family",
    category_orders={"family": family_order},
    hover_data={
        "spi": True, "M": True, "T": True,
        "amortized_s": ":.4f", "family": False,
        "log_walltime_s": False,
    },
    points="all",
    height=500,
    width=900,
    range_y=[df_focus["log_walltime_s"].min() - 0.2,
             df_focus["log_walltime_s"].max() + 0.2],
)

style = {
    "side": "positive",
    "pointpos": -0.35,
    "jitter": 0.22,
    "width": 0.55,
    "meanline_visible": False,
    "box_visible": False,
    "marker": {"size": 6, "opacity": 0.72},
    "opacity": 0.72,
}
for trace in fig.data:
    trace.update(**style)

fig.update_layout(
    yaxis={
        "title": "per-SPI amortized walltime (s, log scale)",
        "tickmode": "array",
        "tickvals": tickvals,
        "ticktext": ticktext,
    },
    xaxis_title="family",
    title=f"Per-family raincloud — M={FOCUS_M}, T={FOCUS_T} snapshot",
    legend_title_text="family",
    margin={"l": 80, "r": 20, "t": 60, "b": 80},
)
fig.show()


### 3c. Per-module raincloud — dots coloured by family

Same module grouping, but each SPI's dot carries its family colour so you can read off the family mix inside each module (e.g. M14 is mostly `basic`; `Mxx` is mostly `causal` + GP variants).

In [ ]:
families_present = [f for f in FAMILY_ORDER if f in focus_pos["family"].values]

exp_min = int(np.floor(focus_pos["log_walltime_s"].min()))
exp_max = int(np.ceil(focus_pos["log_walltime_s"].max()))
y_tickvals = list(range(exp_min, exp_max + 1))
y_ticktext = [f"{10**e:g}" for e in y_tickvals]

# Force a linear x-axis with module labels so violin and scatter x values share
# the same numeric coordinate system. Otherwise plotly auto-treats x as categorical
# (from the violin's string x values) and numeric scatter x can land unpredictably.
m_idx = {m: i for i, m in enumerate(modules_present)}
focus_3c = focus_pos.copy()
focus_3c["module_x"] = focus_3c["module"].map(m_idx)

fig = px.violin(
    focus_3c,
    y="log_walltime_s", x="module_x", color="module",
    color_discrete_map=MODULE_COLORS,
    category_orders={"module": modules_present},
    points=False,
    height=500,
    width=900,
    range_y=[focus_3c["log_walltime_s"].min() - 0.2,
             focus_3c["log_walltime_s"].max() + 0.2],
)

for trace in fig.data:
    trace.update(side="positive", width=0.55,
                 meanline_visible=False, box_visible=False,
                 opacity=0.55, showlegend=False, hoverinfo="skip")

# Family-coloured dots at exactly the raincloud position from 3a:
# offset = pointpos*(width/2), jitter_amp = jitter*(width/2) with pointpos=-0.35,
# jitter=0.22, width=0.55  ->  offset=-0.0963, jitter_amp=0.0605.
_off, _jit = -0.35 * (0.55 / 2), 0.22 * (0.55 / 2)
rng = np.random.default_rng(0)
for fam in families_present:
    sub = focus_3c[focus_3c["family"] == fam]
    if not len(sub):
        continue
    xs = [m_idx[m] + _off + rng.uniform(-_jit, _jit) for m in sub["module"]]
    fig.add_trace(go.Scatter(
        x=xs, y=sub["log_walltime_s"], mode="markers",
        marker={"size": 6, "color": FAMILY_COLORS[fam], "opacity": 0.9,
                "line": {"width": 0.4, "color": "black"}},
        name=fam,
        customdata=list(zip(sub["spi"], sub["module"], sub["amortized_s"])),
        hovertemplate=("spi=%{customdata[0]}<br>module=%{customdata[1]}<br>"
                       f"family={fam}<br>walltime=%{{customdata[2]:.4f}} s<extra></extra>"),
    ))

fig.update_layout(
    xaxis={
        "title": "module",
        "tickmode": "array",
        "tickvals": list(m_idx.values()),
        "ticktext": modules_present,
        "range": [-0.5, len(modules_present) - 0.5],
    },
    yaxis={
        "title": "per-SPI amortized walltime (s, log scale)",
        "tickmode": "array",
        "tickvals": y_tickvals,
        "ticktext": y_ticktext,
    },
    title=f"Per-module raincloud, points coloured by family — M={FOCUS_M}, T={FOCUS_T}",
    legend_title_text="family",
    margin={"l": 80, "r": 20, "t": 60, "b": 80},
)
fig.show()


## 4. Figure group 2 — Cumulative amortized cost vs kept-fraction

Cleaner interactive recreation of `bench/results/analysis/plot_cumulative.png`. Each line is one cell; sorting SPIs cheapest → most expensive, the curve climbs slowly then explodes near 100% — the elbow marks the Pareto front for cutting `pyspi/configs/benchmarked_p<N>.yaml`.

In [ ]:
cells_sorted = sorted(df[["M", "T"]].drop_duplicates().itertuples(index=False),
                      key=lambda r: (r.M, r.T))

fig, ax = plt.subplots(figsize=(8, 5))
for r in cells_sorted:
    sub = df[(df["M"] == r.M) & (df["T"] == r.T)].sort_values("amortized_s")
    costs = sub["amortized_s"].values
    if costs.sum() == 0:
        continue
    frac_kept = np.arange(1, len(costs) + 1) / len(costs) * 100
    cum_cost = np.cumsum(costs) / costs.sum()
    ax.plot(frac_kept, cum_cost, lw=1.2, alpha=0.75, label=f"M={r.M} T={r.T}")
for p in (80, 90, 95):
    ax.axvline(p, color="darkred", lw=1, ls="--", alpha=0.5)
    ax.text(p, 1.01, f"p{p}", ha="center", va="bottom", fontsize=8, color="darkred")
ax.set_xlim(0, 100)
ax.set_xlabel("kept-fraction (%)")
ax.set_ylabel("cumulative amortized cost / total")
ax.set_title("Cumulative amortized cost vs kept-fraction (elbow ≈ cut point)")
ax.legend(fontsize=7, ncol=2, loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Figure group 3 — Scaling

Cell-total amortized walltime vs M (lines per T) and vs T (lines per M), on log-log axes. Ticks are placed at the actual measured M and T values (powers of 2 in M; doubling sequence in T).

In [ ]:
cell_totals = df.groupby(["M", "T"])["amortized_s"].sum().reset_index()

M_TICKS = sorted(cell_totals["M"].unique())
T_TICKS = sorted(cell_totals["T"].unique())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
for T_, sub in cell_totals.groupby("T"):
    ax1.plot(sub["M"], sub["amortized_s"], marker="o", lw=1.5, label=f"T={T_}")
ax1.set_xscale("log", base=2); ax1.set_yscale("log")
ax1.set_xticks(M_TICKS); ax1.set_xticklabels(M_TICKS); ax1.minorticks_off()
ax1.set_xlabel("M"); ax1.set_ylabel("total amortized walltime (s)")
ax1.set_title("Scaling vs M")
ax1.legend(fontsize=8, title="T")
ax1.grid(alpha=0.3, which="both")

for M_, sub in cell_totals.groupby("M"):
    ax2.plot(sub["T"], sub["amortized_s"], marker="o", lw=1.5, label=f"M={M_}")
ax2.set_xscale("log", base=2); ax2.set_yscale("log")
ax2.set_xticks(T_TICKS); ax2.set_xticklabels(T_TICKS); ax2.minorticks_off()
ax2.set_xlabel("T"); ax2.set_ylabel("total amortized walltime (s)")
ax2.set_title("Scaling vs T")
ax2.legend(fontsize=8, title="M")
ax2.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()
